In [1]:
#pip install hf_xet
#pip install langchain langchain-community numpy langchain-huggingface langchain-classic
#pip install -U langchain-text-splitters sentence-transformers unstructured
#pip install libmagic
#pip install bitsandbytes
#pip install llama-cpp-python
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# .\venv\Scripts\activate

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load your code files or documents
loader = DirectoryLoader('./RAGData', glob="**/*.py")
docs = loader.load()

# 2. Split text into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

# 3. Create embeddings and store in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#if used chromdb
#vector_db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db") 
#from langchain_community.vectorstores import Chroma


c:\Users\Yap Zheng Xian\Documents\Programming\Extension\venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Yap Zheng Xian\Documents\Programming\Extension\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Yap Zheng Xian\AppData\Local\Temp\ipykernel_2896\233354464.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = H

In [3]:
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

# 1. Connect to a local folder
db = lancedb.connect("./notebook/RAGData")

# 2. Select your embedding model (LanceDB handles the download/setup)
func = get_registry().get("sentence-transformers").create(
    name="all-MiniLM-L6-v2", # Or use a code-specific model here
    device="cpu" 
)

# 3. Define your table schema
class CodeDocs(LanceModel):
    text: str = func.SourceField()       # The raw code/text
    vector: Vector(func.ndims()) = func.VectorField() # The numbers
    filename: str                        # Metadata

# 4. Create the table
table = db.create_table("code_repo", schema=CodeDocs, mode="overwrite")

# 5. Ingest data (LanceDB embeds 'text' automatically!)
data = [
    {"text": "def hello(): print('world')", "filename": "test.py"},
    {"text": "class Database: pass", "filename": "db.py"}
]
table.add(data)

# 6. Search
results = table.search("How do I print something?").limit(2).to_list()
print(results[0]["text"])

def hello(): print('world')


In [4]:
# 1. Connect to a local folder
db = lancedb.connect("./notebook/RAGData")

# 2. Select your embedding model (LanceDB handles the download/setup)
func = get_registry().get("sentence-transformers").create(
    name="all-MiniLM-L6-v2", # Or use a code-specific model here
    device="cpu" 
)

# 3. Define your table schema
class CodeDocs(LanceModel):
    text: str = func.SourceField()       # The raw code/text
    vector: Vector(func.ndims()) = func.VectorField() # The numbers
    filename: str                        # Metadata

# 4. Create the table
table = db.create_table("code_repo", schema=CodeDocs, mode="overwrite")

# 5. Ingest data (LanceDB embeds 'text' automatically!)
data = [
    {"text": "def hello(): print('world')", "filename": "test.py"},
    {"text": "class Database: pass", "filename": "db.py"}
]
table.add(data)

# 6. Search
results = table.search("How do I print something?").limit(2).to_list()
print(results[0]["text"])




def hello(): print('world')


In [5]:
'''
from langchain.chains import RetrievalQA

# Set up the retriever
retriever = db.as_retriever(
  search_kwargs={"k": 2}
  )

# Create the RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=starcoder_llm,
    chain_type="stuff",
    retriever=retriever
)

# Test it!
query = "How do I implement the login function in this project?"
response = qa_chain.invoke(query)
print(response["result"])
'''

'\nfrom langchain.chains import RetrievalQA\n\n# Set up the retriever\nretriever = db.as_retriever(\n  search_kwargs={"k": 2}\n  )\n\n# Create the RAG chain\nqa_chain = RetrievalQA.from_chain_type(\n    llm=starcoder_llm,\n    chain_type="stuff",\n    retriever=retriever\n)\n\n# Test it!\nquery = "How do I implement the login function in this project?"\nresponse = qa_chain.invoke(query)\nprint(response["result"])\n'

In [6]:
import lancedb
import torch
from langchain_community.vectorstores import LanceDB
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_community.document_loaders import DirectoryLoader

# NEW MODERN IMPORTS (No dot-chains)
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_core.messages import HumanMessage, AIMessage

# ONLY keep this if you specifically need the old 'RetrievalQA' class
# from langchain_classic.chains import RetrievalQA

# --- 1. SETUP EMBEDDINGS ---
# Using a code-specific embedding model for better results with StarCoder2
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# --- 2. LOAD AND CHUNK DATA ---
loader = DirectoryLoader('./MyCodeBase', glob="**/*.py")
docs = loader.load()

# Split using Python-specific logic so functions aren't cut in half
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, 
    chunk_size=1000, 
    chunk_overlap=100
)
chunks = python_splitter.split_documents(docs)

# --- 3. SETUP LANCEDB ---
db = lancedb.connect("./RAGData")
table_name = "code_repo"
# LangChain wrapper for LanceDB

# Check if table already exists
if table_name in db.list_tables():
    # Just load the existing table
    vector_store = LanceDB(connection=db, table_name=table_name, embedding=embeddings)
    print("Loaded existing table!")
else:
    # Create it for the first time
    vector_store = LanceDB.from_documents(chunks, embeddings, connection=db, table_name=table_name)
    print("Created new table!")

# --- 4. SETUP STARCODER2 ---
from transformers import BitsAndBytesConfig
model_id = "bigcode/starcoder2-3b"

# Modern quantization config for your 4GB GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the 3B model (it will only take ~2.5GB of VRAM)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", # Automatically puts it on your GPU
    quantization_config=quantization_config
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)
llm = HuggingFacePipeline(pipeline=pipe)

# --- 5. THE RAG CHAIN ---
# 5.1 Define a prompt
system_prompt = (
    "You are an expert software engineer and Cybersecurity Expert. Below is the source code context "
    "retrieved from the repository. Use it to answer the user's question accurately. "
    "Include code snippets in your answer if they are relevant.\n\n"
    "CODE CONTEXT:\n"
    "----------------------\n"
    "{context}\n"
    "----------------------"
)

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history."
)


contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])


# 5.2 Define the Answer Prompt
system_prompt = (
    "You are a helpful coding assistant. Use the following pieces of retrieved "
    "code context to answer the question. Always mention the filename in your answer."
    "\n\n"
    "CODE CONTEXT:\n"
    "{context}"
)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

from langchain_classic.chains import create_history_aware_retriever
# 5.3 Link the history-aware retriever
# This step "rewrites" the user query to include context from the chat history
history_aware_retriever = create_history_aware_retriever(
    llm, vector_store.as_retriever(), contextualize_q_prompt
)

# 5.4 Create the Final RAG Chain
combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(vector_store.as_retriever(), combine_docs_chain)

# --- 6. RUNNING WITH MEMORY & CITATIONS ---

chat_history = []  # This will store your conversation
def ask_bot(query):
    global chat_history
    
    # Generate Response
    result = rag_chain.invoke({"input": query, "chat_history": chat_history})
    
    # Update History
    chat_history.extend([
        HumanMessage(content=query),
        AIMessage(content=result["answer"]),
    ])
    
    # Print Answer
    print(f"\n[BOT]: {result['answer']}")
    
    # Print Citations
    print("\n[SOURCES]:")
    unique_sources = set(doc.metadata.get('source', 'Unknown') for doc in result["context"])
    for source in unique_sources:
        print(f"- {source}")

def clear_memory():
    global chat_history
    chat_history = []
    print("\n[SYSTEM]: Chat history has been cleared. The bot has 'forgotten' the previous context.")
    
# Example Usage
ask_bot("How do I initialize the database?")



libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


Created new table!


Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



[BOT]: System: You are a helpful coding assistant. Use the following pieces of retrieved code context to answer the question. Always mention the filename in your answer.

CODE CONTEXT:
def get_user_input_field(): """ PURPOSE: Generates a secure HTML input field for usernames.

CYBERSECURITY REQUIREMENT: Enforce a specific input format to prevent SQL Injection and Cross-Site Scripting (XSS) attacks at the browser level.

IMPLEMENTATION EXAMPLE: ```html <input type="text" name="username" pattern="[a-zA-Z]{8}" title="8 character alphanumeric username"> ``` """ # Python code to return or render this field return '<input type="text" name="username" pattern="[a-zA-Z]{8}">'

def file_upload_input_field(): """ PURPOSE: Generates a secure HTML input field for File Upload.

CYBERSECURITY REQUIREMENT: Enforce a specific input file format to prevent wrong file type beign uploaded to server. With limiting the file size.

IMPLEMENTATION EXAMPLE:

```html

#For single file upload

<input type="file"

In [7]:
# Now ask a follow-up question!
ask_bot("Can you show me the code for that?")

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



[BOT]: System: You are a helpful coding assistant. Use the following pieces of retrieved code context to answer the question. Always mention the filename in your answer.

CODE CONTEXT:
def get_user_input_field(): """ PURPOSE: Generates a secure HTML input field for usernames.

CYBERSECURITY REQUIREMENT: Enforce a specific input format to prevent SQL Injection and Cross-Site Scripting (XSS) attacks at the browser level.

IMPLEMENTATION EXAMPLE: ```html <input type="text" name="username" pattern="[a-zA-Z]{8}" title="8 character alphanumeric username"> ``` """ # Python code to return or render this field return '<input type="text" name="username" pattern="[a-zA-Z]{8}">'

def file_upload_input_field(): """ PURPOSE: Generates a secure HTML input field for File Upload.

CYBERSECURITY REQUIREMENT: Enforce a specific input file format to prevent wrong file type beign uploaded to server. With limiting the file size.

IMPLEMENTATION EXAMPLE:

```html

#For single file upload

<input type="file"

In [8]:
import torch
print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Is CUDA available: True
Current device: NVIDIA GeForce RTX 3050 Laptop GPU


In [9]:
print(f"Model memory footprint: {model_id.get_memory_footprint() / 1e9:.2f} GB")
# For StarCoder2-7B, this should be around 5-6 GB.

AttributeError: 'str' object has no attribute 'get_memory_footprint'